# Cuiman Job Result Openers

This section demonbstrates how use `client.open_job_results()` and to enhance 
the method's capabilities by adding a custom `JobResultOpener`.

We use a test server here:

```bash
wraptile run -- wraptile.services.local.testing:service
```

In [1]:
from cuiman import Client
from gavicore.models import ProcessRequest

In [2]:
client = Client()
# client

In [3]:
# client.get_processes()

In [4]:
client.get_process(process_id="simulate_scene")

ProcessDescription(title='Generate scene for testing', description='Simulate a set scene images slices for testing. Creates an xarray dataset with `periodicity` time slices and writes it as Zarr into a temporary location. Requires installed `dask`, `xarray`, and `zarr` packages.', keywords=None, metadata=None, additionalParameters=None, id='simulate_scene', version='0.0.0', jobControlOptions=None, outputTransmission=None, links=None, inputs={'var_names': InputDescription(title='Variable names', description='Comma-separated list of variable names.', keywords=None, metadata=None, additionalParameters=AdditionalParameters(title=None, role=None, href=None, parameters=[AdditionalParameter(name='level', value=['advanced'])]), minOccurs=1, maxOccurs=None, schema_=Schema(field_ref=None, title=None, multipleOf=None, maximum=None, exclusiveMaximum=False, minimum=None, exclusiveMinimum=False, maxLength=None, minLength=0, pattern=None, maxItems=None, minItems=0, uniqueItems=False, maxProperties=No

In [5]:
job_info = client.execute_process(process_id="simulate_scene", request={"inputs": {"output_path": "test.zarr"}})
job_info

JobInfo(processID='simulate_scene', type=<JobType.process: 'process'>, jobID='job_5', status=<JobStatus.accepted: 'accepted'>, message=None, created=datetime.datetime(2026, 3, 4, 12, 3, 5, 557521, tzinfo=TzInfo(0)), started=None, finished=None, updated=None, progress=None, links=None, traceback=None)

In [6]:
try:
    dataset = client.open_job_result(job_info.jobID, timeout=5)
except Exception as e:
    print(f"error: {e}")

error: No job result openers registered


In [7]:
client.get_job_results(job_info.jobID)

JobResults(root={'return_value': InlineOrRefValue(root=InlineValue(root={'href': 'file:///C:/Users/norma/Projects/eozilla/notebooks/test.zarr', 'rel': None, 'type': 'application/zarr', 'hreflang': None, 'title': None}))})

In [8]:
import xarray as xr
from cuiman.api.opener import Opener

class MyZarrOpener(Opener):    
    async def accept(self, ctx):
        print(ctx.output_media_type)
        return ctx.output_media_type == "application/zarr"
    
    async def open(self, ctx):
        return xr.open_zarr(ctx.output_value.href, **ctx.options)

In [9]:
client.config.opener_registry.clear()
client.config.opener_registry.register(MyZarrOpener())

<function cuiman.api.opener.registry.OpenerRegistry.register.<locals>.unregister()>

In [10]:
try:
    dataset = client.open_job_result(job_info.jobID)
except Exception as e:
    print(f"error: {e}")

None
error: No job result opener found for root={'return_value': InlineOrRefValue(root=InlineValue(root={'href': 'file:///C:/Users/norma/Projects/eozilla/notebooks/test.zarr', 'rel': None, 'type': 'application/zarr', 'hreflang': None, 'title': None}))}
